# Multiclass classification example

`bochan.api` の高レベル API を使って、多クラス分類モデルを構築・学習・予測・候補点生成する最小例です。

この notebook では次を扱います。

1. 3 クラスの toy data を作る
2. `task_type="multiclass"` / `model_type="base"` でモデルを学習する
3. class probability と予測 class を確認する
4. notebook 内で簡易 `qMulticlassPredictiveEntropy` を定義し、不確かな点を候補として選ぶ
5. `ask` / `tell` 形式で 1 step 更新する

> 現時点では multiclass 専用 acquisition alias は registry に未登録なので、この notebook では acquisition class を notebook 内で直接定義して `AcquisitionConfig(acqf_cls=...)` に渡します。

In [ ]:
import warnings

import matplotlib.pyplot as plt
import torch
from botorch.acquisition.acquisition import AcquisitionFunction

from bochan.api import (
    AcquisitionConfig,
    BayesianOptimizer,
    FitConfig,
    ModelConfig,
    OptimizeConfig,
)

warnings.filterwarnings("ignore")

torch.manual_seed(0)
dtype = torch.double
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Toy data

2次元入力 `X = [x0, x1]` を 3 つの中心に最も近いクラスへ割り当てる toy problem を作ります。

In [ ]:
def multiclass_target(X: torch.Tensor) -> torch.Tensor:
    """3クラス分類用の toy target。"""

    centers = torch.tensor(
        [
            [0.20, 0.25],
            [0.80, 0.30],
            [0.50, 0.80],
        ],
        dtype=X.dtype,
        device=X.device,
    )
    dist2 = ((X.unsqueeze(-2) - centers) ** 2).sum(dim=-1)
    return dist2.argmin(dim=-1).long()


n_train = 45
train_X = torch.rand(n_train, 2, dtype=dtype, device=device)
train_Y = multiclass_target(train_X)

bounds = torch.tensor(
    [
        [0.0, 0.0],
        [1.0, 1.0],
    ],
    dtype=dtype,
    device=device,
)

print(train_X.shape, train_Y.shape)
print(torch.bincount(train_Y, minlength=3))

In [ ]:
plt.figure(figsize=(5, 4))
plt.scatter(train_X[:, 0].cpu(), train_X[:, 1].cpu(), c=train_Y.cpu(), s=48)
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.xlabel("x0")
plt.ylabel("x1")
plt.title("Initial multiclass training data")
plt.show()

## 2. モデル構築・学習

`ModelConfig(task_type="multiclass", model_type="base")` を使います。

多クラス分類モデルは `SoftmaxLikelihood` を使う variational GP なので、`FitConfig` では `num_epochs` と `lr` を指定します。サンプル notebook では実行時間を短くするため `num_inducing_points=32` にしています。

In [ ]:
model_config = ModelConfig(
    task_type="multiclass",
    model_type="base",
    model_kwargs={
        "num_classes": 3,
        "num_inducing_points": 32,
        "temperature": 1.0,
    },
)

fit_config = FitConfig(
    num_epochs=250,
    lr=0.03,
    batch_size=None,
    verbose=False,
)

bo = BayesianOptimizer(
    model_config=model_config,
    fit_config=fit_config,
    bounds=bounds,
)

bo.fit(train_X, train_Y)

print(type(bo.model).__name__)
print(bo.bundle.metadata)

## 3. 予測確認

`posterior.mean` は class probability として扱える想定です。`argmax` を取れば予測 class になります。

In [ ]:
test_X = torch.tensor(
    [
        [0.20, 0.20],
        [0.85, 0.25],
        [0.50, 0.85],
        [0.50, 0.50],
    ],
    dtype=dtype,
    device=device,
)

with torch.no_grad():
    posterior = bo.model.posterior(test_X)
    probs = posterior.mean
    pred_class = probs.argmax(dim=-1)

print("probs =")
print(probs.detach().cpu())
print("pred_class =", pred_class.detach().cpu().tolist())

In [ ]:
grid_size = 80
x0 = torch.linspace(0, 1, grid_size, dtype=dtype, device=device)
x1 = torch.linspace(0, 1, grid_size, dtype=dtype, device=device)
xx0, xx1 = torch.meshgrid(x0, x1, indexing="xy")
grid_X = torch.stack([xx0.reshape(-1), xx1.reshape(-1)], dim=-1)

with torch.no_grad():
    grid_probs = bo.model.posterior(grid_X).mean
    grid_pred = grid_probs.argmax(dim=-1).reshape(grid_size, grid_size)
    grid_conf = grid_probs.max(dim=-1).values.reshape(grid_size, grid_size)

plt.figure(figsize=(5, 4))
plt.contourf(xx0.cpu(), xx1.cpu(), grid_pred.cpu(), levels=[-0.5, 0.5, 1.5, 2.5], alpha=0.35)
plt.scatter(train_X[:, 0].cpu(), train_X[:, 1].cpu(), c=train_Y.cpu(), s=40, edgecolors="k")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.xlabel("x0")
plt.ylabel("x1")
plt.title("Predicted class region")
plt.show()

plt.figure(figsize=(5, 4))
plt.contourf(xx0.cpu(), xx1.cpu(), grid_conf.cpu(), levels=20)
plt.scatter(train_X[:, 0].cpu(), train_X[:, 1].cpu(), c=train_Y.cpu(), s=40, edgecolors="k")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.xlabel("x0")
plt.ylabel("x1")
plt.title("Max class probability")
plt.colorbar(label="confidence")
plt.show()

## 4. 簡易 multiclass acquisition

ここでは予測クラス確率のエントロピーを acquisition value にします。

`H(y | x) = - sum_c p_c(x) log p_c(x)`

値が大きいほど「どのクラスか不確か」なので、分類境界付近を選びやすくなります。

In [ ]:
class qMulticlassPredictiveEntropy(AcquisitionFunction):
    """多クラス分類用の簡易 predictive entropy acquisition。

    Notes:
        - model.posterior(X).mean が class probability (..., q, C) を返す前提です。
        - q > 1 の場合は batch 内の平均 entropy を返します。
        - 本格的には bochan.acquisition.multiclass などに移して registry 登録するとよいです。
    """

    def __init__(self, model, eps: float = 1e-8) -> None:
        super().__init__(model=model)
        self.eps = float(eps)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        probs = self.model.posterior(X).mean.clamp_min(self.eps)
        entropy = -(probs * probs.log()).sum(dim=-1)
        return entropy.mean(dim=-1)

In [ ]:
acq_config = AcquisitionConfig(
    name="qMulticlassPredictiveEntropy",
    acqf_cls=qMulticlassPredictiveEntropy,
)

opt_config = OptimizeConfig(
    q=3,
    num_restarts=10,
    raw_samples=128,
    sequential=True,
)

candidates, acq_value = bo.candidate(
    acq_config=acq_config,
    opt_config=opt_config,
)

print("candidates =")
print(candidates.detach().cpu())
print("acq_value =")
print(acq_value.detach().cpu())

In [ ]:
with torch.no_grad():
    cand_probs = bo.model.posterior(candidates).mean
    cand_entropy = -(cand_probs.clamp_min(1e-8) * cand_probs.clamp_min(1e-8).log()).sum(dim=-1)

print("candidate probs =")
print(cand_probs.detach().cpu())
print("candidate entropy =", cand_entropy.detach().cpu())

In [ ]:
plt.figure(figsize=(5, 4))
plt.contourf(xx0.cpu(), xx1.cpu(), grid_conf.cpu(), levels=20)
plt.scatter(train_X[:, 0].cpu(), train_X[:, 1].cpu(), c=train_Y.cpu(), s=40, edgecolors="k", label="train")
plt.scatter(candidates[:, 0].detach().cpu(), candidates[:, 1].detach().cpu(), marker="*", s=180, label="candidate")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.xlabel("x0")
plt.ylabel("x1")
plt.title("Entropy-based candidates")
plt.legend()
plt.colorbar(label="confidence")
plt.show()

## 5. ask / tell 形式で 1 step 更新

実験・Web アプリで使う場合は、まず `ask` で候補点だけ生成し、実験結果が返ってきた後に `tell` で追加します。

In [ ]:
ask_X, ask_value = bo.ask(
    acq_config=acq_config,
    opt_config=OptimizeConfig(q=1, num_restarts=10, raw_samples=128),
)

new_Y = multiclass_target(ask_X)

print("ask_X =", ask_X.detach().cpu())
print("new_Y =", new_Y.detach().cpu())

bo.tell(
    new_X=ask_X,
    new_Y=new_Y,
    refit=True,
    fit_config=FitConfig(num_epochs=150, lr=0.03, verbose=False),
)

print("updated train_X:", bo.train_X.shape)
print("updated train_Y:", bo.train_Y.shape)

## 6. mixed 入力版の最小例

カテゴリ列を含む場合は `cat_dims` を指定します。`BayesianOptimizer.candidate(...)` 側で mixed optimizer へ自動解決されます。

In [ ]:
n_mixed = 36
X_cont = torch.rand(n_mixed, 2, dtype=dtype, device=device)
X_cat = torch.randint(0, 3, (n_mixed, 1), device=device).to(dtype)
mixed_X = torch.cat([X_cont, X_cat], dim=-1)
mixed_Y = (multiclass_target(X_cont) + X_cat.squeeze(-1).long()) % 3

mixed_bounds = torch.tensor(
    [
        [0.0, 0.0, 0.0],
        [1.0, 1.0, 2.0],
    ],
    dtype=dtype,
    device=device,
)

mixed_bo = BayesianOptimizer(
    model_config=ModelConfig(
        task_type="multiclass",
        model_type="base",
        cat_dims=[2],
        model_kwargs={
            "num_classes": 3,
            "num_inducing_points": 24,
        },
    ),
    fit_config=FitConfig(num_epochs=150, lr=0.03, verbose=False),
    bounds=mixed_bounds,
)

mixed_bo.fit(mixed_X, mixed_Y)

mixed_candidates, mixed_acq_value = mixed_bo.candidate(
    acq_config=AcquisitionConfig(
        name="qMulticlassPredictiveEntropy",
        acqf_cls=qMulticlassPredictiveEntropy,
    ),
    opt_config=OptimizeConfig(q=2, num_restarts=8, raw_samples=64, sequential=True),
)

print(mixed_candidates.detach().cpu())
print(mixed_acq_value.detach().cpu())

## 7. 次に実装するとよいこと

この notebook の `qMulticlassPredictiveEntropy` が実運用で有用なら、次は次のように本体実装へ移すとよいです。

- `src/bochan/acquisition/multiclass/active_learning.py`
- `qMulticlassPredictiveEntropy`
- `qMulticlassProbabilityVariance`
- `qMulticlassMarginUncertainty`
- `qMulticlassBALD`
- `src/bochan/api/registry/acquisition.py` への registry 登録

そうすると `AcquisitionConfig(name="entropy")` や `AcquisitionConfig(name="qMulticlassPredictiveEntropy")` のように、notebook 内で class 定義せず使えるようになります。